# Adapting BERT for Named Entity Recognition (CoNLL-2003)

**Course assignment: Feature-based adaptation vs. full fine-tuning of `bert-base-cased`**

This notebook compares two ways of adapting a pretrained transformer to **Named Entity Recognition (NER)**, the task of tagging each word in a sentence as a *Person*, *Organisation*, *Location*, *Miscellaneous* entity, or *nothing* (`O`).

| | Model 1: Feature-based | Model 2: Full fine-tuning |
|---|---|---|
| BERT body | **frozen** (0 trainable parameters) | trained end to end |
| Classifier | scikit-learn Logistic Regression | linear head of `BertForTokenClassification` |
| Learning rates | n/a | **2e-5 encoder / 1e-3 head** |

### Contents
1. Environment & setup (fixed random seed)
2. Data loading: CoNLL-2003 and the `bert-base-cased` tokenizer
3. Tokenization & the sub-word alignment rule (+ sanity check)
4. Model 1: frozen BERT features + Logistic Regression
5. Model 2: full fine-tuning with two learning rates
6. Evaluation with `seqeval` (entity-level Precision / Recall / F1)
7. Publishing the fine-tuned model to the Hugging Face Hub

> **Hardware.** Model 2 uses mixed precision (`fp16=True`), which needs a CUDA GPU. On an RTX 4060 (8 GB) the whole notebook runs in about 6 minutes.

---
## 1. Environment & Setup

Required packages (install once):

```bash
pip install torch transformers datasets accelerate seqeval scikit-learn huggingface_hub pandas
```

**Why fix a random seed?** Several parts of this pipeline are random: the initial weights of the new classification head, the order in which training batches are shuffled, dropout, and the sub-sampling of tokens for Model 1. Fixing the seed of every random number generator (Python, NumPy, PyTorch) means that running the notebook again gives the same numbers, so the two models are compared under identical conditions.

In [ ]:
import random
import time

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import (
    AutoModel,
    AutoTokenizer,
    BertForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
    pipeline,
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from seqeval.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)

In [ ]:
SEED = 42


def set_seed(seed):
    # Fix every random number generator used in this notebook.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)

MODEL_NAME = "bert-base-cased"  # cased: capitalisation is a strong signal for names
IGNORE_INDEX = -100             # label value that the loss function skips (see Section 3)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## 2. Data Loading

### The CoNLL-2003 dataset
CoNLL-2003 is the standard English NER benchmark: Reuters news articles from 1996, **already split into words**, with one tag per word in the **IOB2 format**:

- `B-XXX`: the **B**eginning of an entity of type `XXX`
- `I-XXX`: **I**nside (a continuation of) the same entity
- `O`: **O**utside any entity

For example, *"Peter Blackburn"* is tagged `B-PER I-PER`. The four entity types are `PER`, `ORG`, `LOC` and `MISC`, which gives 9 labels in total.

> **Note on loading.** Recent versions of `datasets` (4.0+) no longer run dataset *loading scripts*, and the original `conll2003` repository is one. If the plain name fails, we load `eriktks/conll2003` at revision `refs/convert/parquet`. This is the Hub's automatic Parquet export of **the same dataset**, with the same splits and label names.

In [ ]:
try:
    raw_datasets = load_dataset("conll2003")
except Exception as err:
    print(f"load_dataset('conll2003') failed ({type(err).__name__}); "
          "using the Parquet export of the same dataset instead.")
    raw_datasets = load_dataset("eriktks/conll2003", revision="refs/convert/parquet")

raw_datasets

In [ ]:
# The integer -> tag-name mapping is stored in the dataset's metadata.
label_list = raw_datasets["train"].features["ner_tags"].feature.names
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

print("Labels:", label_list)

# One training sentence, word by word.
example = raw_datasets["train"][0]
pd.DataFrame({
    "word": example["tokens"],
    "tag_id": example["ner_tags"],
    "tag": [label_list[t] for t in example["ner_tags"]],
}).T

### The `bert-base-cased` tokenizer
BERT does not read whole words. Its **WordPiece** tokenizer has a fixed vocabulary of about 29k pieces, and a word that is not in the vocabulary is split into smaller **sub-tokens**. Continuation pieces are marked with `##`. We load the *fast* (Rust) tokenizer because it provides `word_ids()`, which records the original word that each sub-token came from. We need that mapping in the next section.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Fast tokenizer:", tokenizer.is_fast)

for word in ["Washington", "lamb", "BRUSSELS", "Blackburn"]:
    print(f"{word:<12} -> {tokenizer.tokenize(word)}")

---
## 3. Tokenization & the Alignment Rule

### The problem
Labels exist **per word**, but BERT produces one output vector **per sub-token**. After tokenization the sequences no longer line up:

```
words     :  EU      rejects  German  call  to  boycott  British  lamb       .
tags      :  B-ORG   O        B-MISC  O     O   O        B-MISC   O          O
sub-tokens:  [CLS] EU rejects German call to boycott British la ##mb . [SEP]
```

### The rule used in this assignment
| Position | Label |
|---|---|
| **first** sub-token of a word | the word's **true label** |
| continuation sub-tokens (`##mb`) | `-100` |
| special tokens `[CLS]`, `[SEP]` | `-100` |
| padding `[PAD]` | `-100` |

This way every word is counted **exactly once**, on its first sub-token.

### Why `-100`? How `CrossEntropyLoss` uses it
PyTorch's `nn.CrossEntropyLoss` has a parameter `ignore_index`, which is **`-100` by default**. Any position whose target equals `ignore_index`:

- adds **nothing to the loss**, and
- therefore sends **no gradient** back through the network.

The loss is averaged **only over positions that carry a real label**:

$$\mathcal{L} = -\frac{1}{|V|}\sum_{i \in V}\log p_\theta(y_i \mid x), \qquad V = \{\, i : y_i \neq -100 \,\}$$

Without this mask we would have two bad options:
1. **Give `[CLS]`/`[SEP]`/`[PAD]` a real label such as `O`.** The model would then be trained to predict `O` for padding, which is meaningless, and because `O` already makes up about 83% of the words, the class imbalance would get worse.
2. **Copy the word's tag to every sub-token.** Long words split into many pieces (`BR ##US ##SE ##LS`) would count several times in the loss and in the metrics, so rare words would carry more weight than common ones.

Using `-100` keeps a clean **one word = one prediction** correspondence, both during training and during evaluation.

In [ ]:
# A small demonstration: -100 positions are simply left out of the loss.
loss_fn = torch.nn.CrossEntropyLoss()           # ignore_index = -100 by default
logits = torch.randn(5, len(label_list))        # 5 positions, 9 classes
targets = torch.tensor([3, -100, 0, -100, 5])   # 2 positions are masked

loss_with_mask = loss_fn(logits, targets)
valid = targets != IGNORE_INDEX
loss_only_valid = loss_fn(logits[valid], targets[valid])

print(f"loss over all 5 positions (with -100)   : {loss_with_mask:.6f}")
print(f"loss over the 3 labelled positions only : {loss_only_valid:.6f}")

### Implementing the alignment
`tokenizer(..., is_split_into_words=True)` tells the tokenizer that the input is already split into words. `word_ids()` then returns, for every sub-token, the index of the word it came from, or `None` for special tokens. A sub-token is the **first** piece of its word when its `word_id` differs from the previous one.

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(examples["tokens"], is_split_into_words=True, truncation=True)

    all_labels = []
    for i, word_tags in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        labels = []
        previous_word_id = None
        for word_id in word_ids:
            if word_id is None:                   # [CLS] or [SEP]
                labels.append(IGNORE_INDEX)
            elif word_id != previous_word_id:     # first sub-token of a word
                labels.append(word_tags[word_id])
            else:                                 # continuation sub-token (##...)
                labels.append(IGNORE_INDEX)
            previous_word_id = word_id
        all_labels.append(labels)

    tokenized["labels"] = all_labels
    return tokenized


tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)
tokenized_datasets

### Padding and the data collator
Sentences have different lengths, but a batch must be a rectangular tensor. `DataCollatorForTokenClassification` pads every sequence in a batch to the length of the longest one (**dynamic padding**). It pads `input_ids` with `[PAD]` and **`labels` with `-100`** (its `label_pad_token_id`), so padding is ignored by the loss too.

### Sanity check
Before training anything, we print one real batch as the models will see it, to confirm that:
1. only the **first** sub-token of each word has a real label,
2. continuation pieces, `[CLS]`, `[SEP]` **and padding** are all `-100`.

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

loader = DataLoader(tokenized_datasets["train"], batch_size=4, collate_fn=data_collator)
batch = next(iter(loader))

print("SANITY CHECK: one collated training batch")
print("input_ids      shape:", tuple(batch["input_ids"].shape))
print("attention_mask shape:", tuple(batch["attention_mask"].shape))
print("labels         shape:", tuple(batch["labels"].shape))

for row in range(3):
    tokens = tokenizer.convert_ids_to_tokens(batch["input_ids"][row])
    labels = batch["labels"][row].tolist()
    print(f"\n--- sentence {row} ---")
    print(f"{'pos':>3} | {'token':<12} | {'label_id':>8} | {'label':<7} | role")
    n_pad = 0
    for pos, (token, label) in enumerate(zip(tokens, labels)):
        if token == tokenizer.pad_token:
            n_pad += 1
            continue
        if label != IGNORE_INDEX:
            name, role = label_list[label], "first sub-token -> TRUE LABEL"
        elif token in tokenizer.all_special_tokens:
            name, role = "-", "special token   -> ignored"
        else:
            name, role = "-", "continuation    -> ignored"
        print(f"{pos:>3} | {token:<12} | {label:>8} | {name:<7} | {role}")
    print(f"    + {n_pad} [PAD] positions")

# Every padding position must carry -100.
pad_labels = batch["labels"][batch["attention_mask"] == 0]
assert (pad_labels == IGNORE_INDEX).all()
print(f"\nOK: all {pad_labels.numel()} padding positions in this batch are labelled -100.")

In [ ]:
# Corpus-level check: the number of labelled positions must equal the number of words.
n_words = sum(len(tags) for tags in raw_datasets["train"]["ner_tags"])
n_subtokens = sum(len(ids) for ids in tokenized_datasets["train"]["input_ids"])
n_labelled = sum(sum(l != IGNORE_INDEX for l in labels)
                 for labels in tokenized_datasets["train"]["labels"])

print(f"words in the training set    : {n_words:,}")
print(f"sub-tokens (incl. CLS/SEP)   : {n_subtokens:,}")
print(f"positions with a true label  : {n_labelled:,}")
print(f"positions set to -100        : {n_subtokens - n_labelled:,}")
assert n_labelled == n_words, "every word should have exactly one labelled sub-token"
print("OK: exactly one labelled sub-token per word.")

---
## 4. Model 1: Feature-Based Adaptation (baseline)

### Idea
Here BERT is used only as a **fixed feature extractor**, and none of its weights change:

1. **Freeze** every BERT parameter (`requires_grad = False`), leaving **0 trainable parameters** in BERT.
2. Run the sentences through BERT once and take `last_hidden_state`: a **768-dimensional contextual vector** for every sub-token.
3. Keep only the vectors at labelled positions (the `-100` mask from Section 3), which gives one vector per word.
4. Train a **classical scikit-learn classifier** (multinomial Logistic Regression) that maps each vector to one of the 9 tags.

### Why do this?
- **Cheap:** BERT runs only in inference mode (no gradients, no optimizer state), and the classifier trains in seconds on a CPU.
- **A baseline:** it measures how much NER information the *pretrained* representations already contain. The gap to Model 2 then shows how much the full fine-tuning adds.

**Limitation:** each word is classified **independently**. The classifier never sees the tag of the previous word, so it can produce invalid sequences such as `O I-PER`. The BERT features also stay general-purpose rather than NER-specific.

In [ ]:
bert_encoder = AutoModel.from_pretrained(MODEL_NAME)  # the BERT body without any task head

for param in bert_encoder.parameters():
    param.requires_grad = False
bert_encoder.eval().to(device)

total_params = sum(p.numel() for p in bert_encoder.parameters())
trainable_params = sum(p.numel() for p in bert_encoder.parameters() if p.requires_grad)
print(f"BERT parameters: {total_params:,}  |  trainable: {trainable_params:,}")
assert trainable_params == 0, "The BERT body must be completely frozen for Model 1."

### Extracting `last_hidden_state`
For every batch we run the frozen encoder, then use the label mask to select the vectors of labelled positions. We also record which sentence each vector came from (`sentence_ids`), so we can rebuild whole sentences later. `seqeval` scores **entities**, and an entity can span several words, so it needs complete sentences.

In [ ]:
@torch.no_grad()  # no gradients needed: BERT is frozen
def extract_features(dataset, batch_size=64):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=data_collator)
    features, labels, sentence_ids = [], [], []
    sentences_seen = 0

    for batch in loader:
        batch_labels = batch.pop("labels")
        inputs = {name: tensor.to(device) for name, tensor in batch.items()}

        hidden = bert_encoder(**inputs).last_hidden_state.cpu()  # (batch, seq_len, 768)

        mask = batch_labels != IGNORE_INDEX                      # labelled positions only
        features.append(hidden[mask].numpy())
        labels.append(batch_labels[mask].numpy())
        row_index = mask.nonzero()[:, 0]                         # which sentence in the batch
        sentence_ids.append((row_index + sentences_seen).numpy())
        sentences_seen += batch_labels.shape[0]

    return np.concatenate(features), np.concatenate(labels), np.concatenate(sentence_ids)


start = time.time()
X_train, y_train, _ = extract_features(tokenized_datasets["train"])
X_val, y_val, sent_val = extract_features(tokenized_datasets["validation"])
X_test, y_test, sent_test = extract_features(tokenized_datasets["test"])
print(f"Feature extraction took {time.time() - start:.1f}s")
print(f"X_train {X_train.shape} | X_val {X_val.shape} | X_test {X_test.shape}")

### Training the Logistic Regression
- **`StandardScaler`** puts every one of the 768 features on the same scale (mean 0, variance 1), which helps the solver converge.
- **Sub-sampling:** there are about 204k training words. We fit on a random sample of **120,000** of them (fixed by the seed) so the fit takes about 20-30 seconds. More data barely changes the result for a linear model on 768 features.

In [ ]:
MAX_TRAIN_TOKENS = 120_000

rng = np.random.default_rng(SEED)
sample = rng.choice(len(X_train), size=MAX_TRAIN_TOKENS, replace=False)

classifier = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=300, random_state=SEED),
)

start = time.time()
classifier.fit(X_train[sample], y_train[sample])
print(f"Logistic Regression trained on {MAX_TRAIN_TOKENS:,} word vectors in {time.time() - start:.1f}s")

In [ ]:
def to_sentences(sentence_ids, tag_ids):
    # Regroup a flat list of per-word tags into one list of tag names per sentence.
    sentences = {}
    for sent_id, tag_id in zip(sentence_ids, tag_ids):
        sentences.setdefault(sent_id, []).append(label_list[tag_id])
    return [sentences[k] for k in sorted(sentences)]


m1_val_true = to_sentences(sent_val, y_val)
m1_val_pred = to_sentences(sent_val, classifier.predict(X_val))
m1_test_true = to_sentences(sent_test, y_test)
m1_test_pred = to_sentences(sent_test, classifier.predict(X_test))

print("Example sentence (test):")
print("  gold:", m1_test_true[0])
print("  pred:", m1_test_pred[0])

In [ ]:
# Free GPU memory before loading Model 2.
del bert_encoder, X_train, y_train
torch.cuda.empty_cache()

---
## 5. Model 2: Full Fine-Tuning

### Idea
`BertForTokenClassification` = the pretrained BERT encoder + a **new linear layer** (768 → 9) on top of every sub-token vector. Here we train **all ~108M parameters** together with the loss from Section 3, so the representations themselves adapt to NER.

### Two learning rates (discriminative fine-tuning)
The two parts of the model start from very different places:

| Part | Starts as | Learning rate | Why |
|---|---|---|---|
| Encoder (`bert.*`) | pretrained, already very good | **2e-5** (small) | large updates would erase what was learned in pretraining ("catastrophic forgetting") |
| Head (`classifier.*`) | **random** weights | **1e-3** (large, 50×) | it must learn from scratch, and with 2e-5 it would learn too slowly in 3 epochs |

We implement this with **two parameter groups** in one `AdamW` optimizer and pass that optimizer to the `Trainer`.

### Other training settings
- **`fp16=True` (mixed precision):** most operations run in 16-bit floats on the GPU's tensor cores, which is about 2× faster and uses less memory. The `Trainer` applies loss scaling automatically so small gradients do not underflow.
- **Linear schedule with 10% warm-up:** the learning rates rise from 0 during the first 10% of steps, then decay linearly to 0. The warm-up stops the randomly initialised head from sending large, noisy gradients into the encoder at the start.
- **Evaluation on the validation set after every epoch**, keeping the checkpoint with the best entity-level F1.

In [ ]:
set_seed(SEED)  # the classifier head is randomly initialised: make it reproducible

model = BertForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

In [ ]:
ENCODER_LR = 2e-5
HEAD_LR = 1e-3

encoder_params = [p for name, p in model.named_parameters() if name.startswith("bert.")]
head_params = [p for name, p in model.named_parameters() if not name.startswith("bert.")]

optimizer = torch.optim.AdamW(
    [
        {"params": encoder_params, "lr": ENCODER_LR},  # group 0: pretrained encoder
        {"params": head_params, "lr": HEAD_LR},        # group 1: new classifier head
    ],
    weight_decay=0.01,
)

n_encoder = sum(p.numel() for p in encoder_params)
n_head = sum(p.numel() for p in head_params)
print(f"Encoder: {n_encoder:>12,} parameters @ lr = {ENCODER_LR}")
print(f"Head   : {n_head:>12,} parameters @ lr = {HEAD_LR}")
print(f"Total trainable: {n_encoder + n_head:,}")

### Metric used during training
At each evaluation the `Trainer` gives us raw logits of shape `(sentences, sub-tokens, 9)`. We take the `argmax`, **drop every `-100` position** (the same mask as before, so we score one prediction per word), convert the ids to tag names, and let `seqeval` compute entity-level scores. The helper `align_predictions` is reused in Section 6.

In [ ]:
def align_predictions(logits, label_ids):
    # Convert logits + gold ids into per-sentence lists of tag names, skipping -100 positions.
    predictions = np.argmax(logits, axis=-1)
    true_sentences, pred_sentences = [], []
    for pred_row, gold_row in zip(predictions, label_ids):
        keep = gold_row != IGNORE_INDEX
        true_sentences.append([label_list[t] for t in gold_row[keep]])
        pred_sentences.append([label_list[p] for p in pred_row[keep]])
    return true_sentences, pred_sentences


def compute_metrics(eval_pred):
    logits, label_ids = eval_pred
    y_true, y_pred = align_predictions(logits, label_ids)
    return {
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="bert-ner-checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    fp16=True,                     # mixed-precision training (requires a CUDA GPU)
    lr_scheduler_type="linear",
    warmup_steps=0.1,              # a value < 1 means a ratio: warm up during 10% of the steps
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    seed=SEED,
    report_to="none",
)
# Note: TrainingArguments.learning_rate is NOT used here, because we pass our own
# optimizer (with its two learning rates). The Trainer only adds the LR scheduler.

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None),  # None -> Trainer builds the linear warm-up schedule
)

In [ ]:
start = time.time()
train_result = trainer.train()
print(f"Training finished in {(time.time() - start) / 60:.1f} min "
      f"(final training loss {train_result.training_loss:.4f})")

In [ ]:
# Learning curve: validation scores after each epoch.
history = [log for log in trainer.state.log_history if "eval_f1" in log]
pd.DataFrame(history)[["epoch", "eval_loss", "eval_precision", "eval_recall", "eval_f1"]].round(4)

### Saving the fine-tuned model
Because of `load_best_model_at_end=True`, `trainer.model` now holds the epoch with the best validation F1. We save it with its tokenizer to `bert-ner-final/`. This folder has only what is needed to use the model (weights, config, tokenizer), about 430 MB. The checkpoints in `bert-ner-checkpoints/` also contain the optimizer state, so they can be deleted once this is saved.

The model can be loaded again later with `BertForTokenClassification.from_pretrained("bert-ner-final")`.

In [ ]:
FINAL_MODEL_DIR = "bert-ner-final"

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f"Fine-tuned model and tokenizer saved to '{FINAL_MODEL_DIR}/'")

---
## 6. Evaluation & Metrics

### Why not accuracy?
About **83% of CoNLL-2003 words are tagged `O`**. A useless model that outputs `O` everywhere would already get 83% token accuracy while finding **zero** entities. Token accuracy mostly measures how well the model handles the easy majority class.

### Entity-level scoring with `seqeval`
`seqeval` first reads the IOB tags into **entity spans** (e.g. `B-PER I-PER` → one `PER` entity spanning 2 words), then compares spans:

- An entity counts as correct **only if both its boundaries and its type match exactly**. Finding "Peter" but missing "Blackburn" counts as one wrong prediction *and* one missed entity.
- **Precision** = correct predicted entities / all predicted entities (*how many of our predictions are right?*)
- **Recall** = correct predicted entities / all gold entities (*how many real entities did we find?*)
- **F1** = harmonic mean of the two. This is the standard CoNLL-2003 metric.

Both models are scored in exactly the same way: one prediction per word (first sub-token), grouped by sentence, on the **held-out test set**.

In [ ]:
# Model 2 predictions on the validation and test sets.
m2_val_true, m2_val_pred = align_predictions(*trainer.predict(tokenized_datasets["validation"])[:2])
m2_test_true, m2_test_pred = align_predictions(*trainer.predict(tokenized_datasets["test"])[:2])

# Both models must be scored on exactly the same gold tags.
assert m1_test_true == m2_test_true and m1_val_true == m2_val_true


def entity_scores(y_true, y_pred):
    # float(): seqeval returns NumPy numbers, plain floats are easier to save/print
    return {
        "Precision": float(precision_score(y_true, y_pred)),
        "Recall": float(recall_score(y_true, y_pred)),
        "F1": float(f1_score(y_true, y_pred)),
        "Token accuracy (secondary)": float(accuracy_score(y_true, y_pred)),
    }


results = pd.DataFrame({
    ("Validation", "Model 1: frozen BERT + LogReg"): entity_scores(m1_val_true, m1_val_pred),
    ("Validation", "Model 2: full fine-tuning"): entity_scores(m2_val_true, m2_val_pred),
    ("Test", "Model 1: frozen BERT + LogReg"): entity_scores(m1_test_true, m1_test_pred),
    ("Test", "Model 2: full fine-tuning"): entity_scores(m2_test_true, m2_test_pred),
}).T.round(4)
results

In [ ]:
print("=== Model 1: frozen BERT + Logistic Regression (test) ===")
print(classification_report(m1_test_true, m1_test_pred, digits=4))
print("=== Model 2: full fine-tuning (test) ===")
print(classification_report(m2_test_true, m2_test_pred, digits=4))

gain = f1_score(m2_test_true, m2_test_pred) - f1_score(m1_test_true, m1_test_pred)
print(f"Test F1 gain from full fine-tuning: {gain:+.4f}")

### Trying the fine-tuned model on new text
The Hugging Face `pipeline` with `aggregation_strategy="simple"` merges sub-tokens back into whole words and entities.

In [ ]:
ner = pipeline(
    "token-classification",
    model=trainer.model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=0 if device.type == "cuda" else -1,
)

for entity in ner("Angela Merkel met Tim Cook at the Apple offices in Berlin during the European Council summit."):
    print(f"{entity['word']:<20} {entity['entity_group']:<5} score={entity['score']:.3f}")

### Discussion
Read the results table and the per-entity reports above with these points in mind:

1. **Frozen features vs. fine-tuning.** The gap in F1 between the two models measures how much the full fine-tuning adds. The frozen `last_hidden_state` vectors were trained for masked-language modelling, not NER. Fine-tuning adapts all 12 layers to the task.
2. **Accuracy vs. entity F1.** Compare the token accuracy of each model with its entity-level F1. Because most words are `O`, accuracy stays high even when many entities are missed. This is why entity-level `seqeval` is the right metric.
3. **Per-entity pattern.** Check which entity types are easiest and which are hardest. `MISC` is a mixed category (nationalities, events, titles...) with the least consistent annotation, so we expect it to be the weakest.
4. **Boundary errors in the baseline.** Logistic Regression tags each word independently and never sees the neighbouring tags, so it can produce inconsistent spans such as `O I-ORG`. `seqeval` counts each of these as a wrong entity.
5. **Cost.** Model 1 needs one forward pass through BERT plus a quick CPU fit. Model 2 needs GPU training and stores a full copy of BERT (~430 MB). Use the timings printed above to compare them.

---
## 7. Publishing to the Hugging Face Hub

The final step publishes the fine-tuned model, its tokenizer and a **Model Card** (the repository's `README.md`, describing the model, its data, its results and its limitations) on the Hugging Face Hub, and gives the user **`Dexterg83`** access.

**Before running:** log in with a token that has **write** permission (https://huggingface.co/settings/tokens):
```bash
hf auth login          # or set the HF_TOKEN environment variable
```

**How access is granted.** The Hub API can only approve a *specific user* on a **gated** repository. So we create the repo, set gating to `"manual"` **before uploading any files**, and approve `Dexterg83` with `grant_access`. Anyone can see the repository page, but only approved users can download the weights.

In [ ]:
from huggingface_hub import EvalResult, HfApi, ModelCard, ModelCardData

REPO_NAME = "bert-base-cased-conll2003-ner"
COLLABORATOR = "Dexterg83"

api = HfApi()
username = api.whoami()["name"]      # fails here if you are not logged in
repo_id = f"{username}/{REPO_NAME}"
print("Target repository:", repo_id)

In [ ]:
# ---- Model Card -----------------------------------------------------------
test_scores = entity_scores(m2_test_true, m2_test_pred)
baseline_scores = entity_scores(m1_test_true, m1_test_pred)

card_data = ModelCardData(
    language="en",
    license="apache-2.0",
    library_name="transformers",
    pipeline_tag="token-classification",
    base_model=MODEL_NAME,
    datasets=["conll2003"],
    tags=["token-classification", "ner", "bert", "conll2003"],
    model_name=REPO_NAME,
    eval_results=[
        EvalResult(
            task_type="token-classification", task_name="Named Entity Recognition",
            dataset_type="conll2003", dataset_name="CoNLL-2003", dataset_split="test",
            metric_type=metric.lower(), metric_name=metric, metric_value=round(test_scores[metric], 4),
        )
        for metric in ["Precision", "Recall", "F1"]
    ],
)

card_text = f'''---
{card_data.to_yaml()}
---

# {REPO_NAME}

[`{MODEL_NAME}`](https://huggingface.co/{MODEL_NAME}) fine-tuned for **Named Entity Recognition**
on **CoNLL-2003** (entity types PER, ORG, LOC, MISC). University assignment comparing
feature-based adaptation with full fine-tuning.

## Results (CoNLL-2003 test set, entity-level `seqeval`)

| Model | Precision | Recall | F1 |
|---|---|---|---|
| Frozen BERT + Logistic Regression (baseline) | {baseline_scores["Precision"]:.4f} | {baseline_scores["Recall"]:.4f} | {baseline_scores["F1"]:.4f} |
| **This model (full fine-tuning)** | **{test_scores["Precision"]:.4f}** | **{test_scores["Recall"]:.4f}** | **{test_scores["F1"]:.4f}** |

## Training procedure
- Sub-word alignment: the label goes on the first sub-token of each word only; continuation
  sub-tokens, special tokens and padding are set to `-100` (ignored by the loss).
- AdamW with **two parameter groups**: encoder lr = {ENCODER_LR}, classification head lr = {HEAD_LR};
  weight decay 0.01; linear schedule with 10% warm-up.
- {training_args.num_train_epochs:g} epochs, batch size {training_args.per_device_train_batch_size},
  `fp16=True`, seed {SEED}; best epoch selected on validation F1.

## Usage
```python
from transformers import pipeline
ner = pipeline("token-classification", model="{repo_id}", aggregation_strategy="simple")
ner("Angela Merkel visited Microsoft in Redmond.")
```

## Limitations
- Trained on 1996 English Reuters news: expect lower accuracy on social media, other domains,
  other languages, and entities that appeared after 1996.
- The model is case-sensitive: all-lowercase or ALL-CAPS text degrades results.
- `MISC` is the weakest class.
- Not intended for automated decisions about people without human review.
'''

model_card = ModelCard(card_text)
model_card.save("MODEL_CARD.md")   # local copy for the report
print(card_text[:1500])

In [ ]:
# ---- 1) Create the repository and restrict access before uploading --------
api.create_repo(repo_id, repo_type="model", exist_ok=True)
api.update_repo_settings(repo_id, gated="manual")

# ---- 2) Upload the fine-tuned model, the tokenizer and the Model Card -----
trainer.model.push_to_hub(repo_id, commit_message="Add fine-tuned bert-base-cased NER model")
tokenizer.push_to_hub(repo_id, commit_message="Add tokenizer")
model_card.push_to_hub(repo_id, commit_message="Add model card")

# ---- 3) Grant access to the reviewer ---------------------------------------
try:
    api.grant_access(repo_id, COLLABORATOR)
    print(f"Access granted to '{COLLABORATOR}'.")
except Exception as err:
    # e.g. the user already has access
    print(f"Could not grant access automatically ({err}).\n"
          f"Do it by hand: https://huggingface.co/{repo_id}/settings -> Gated user access.")

print(f"Published: https://huggingface.co/{repo_id}")

---
## Summary

| Step | What we did | Key point |
|---|---|---|
| Setup | Fixed seed 42 everywhere | Reproducible comparison |
| Data | CoNLL-2003 + `bert-base-cased` WordPiece | Labels are per word, BERT works per sub-token |
| Alignment | Label on the first sub-token only, `-100` elsewhere | `CrossEntropyLoss(ignore_index=-100)` skips the masked positions |
| Model 1 | Frozen BERT → `last_hidden_state` → Logistic Regression | 0 trainable BERT parameters, cheap baseline |
| Model 2 | `BertForTokenClassification`, `Trainer`, `fp16` | Two learning rates: 2e-5 encoder, 1e-3 head |
| Evaluation | `seqeval` entity-level P / R / F1 | Accuracy is misleading when 83% of words are `O` |
| Hub | `push_to_hub` + Model Card + gated access for `Dexterg83` | Shareable, documented deliverable |